In [140]:
import numpy as np
import torch
import torch.nn as nn

In [141]:
class LSTM:

    def __init__(self,input_size,hidden_size,output_size):

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        concat_size = input_size + hidden_size

        # Gates

        self.Wf = np.random.randn(hidden_size,concat_size) * 0.1

        self.Wi = np.random.randn(hidden_size,concat_size) * 0.1

        self.Wg = np.random.randn(hidden_size,concat_size) * 0.1

        self.Wo = np.random.randn(hidden_size,concat_size) * 0.1

        self.bf = np.zeros((hidden_size,1))
        self.bi = np.zeros((hidden_size,1))
        self.bg = np.zeros((hidden_size,1))
        self.bo = np.zeros((hidden_size,1))

        # Output layer

        self.Why = np.random.randn(output_size,hidden_size) * 0.1

        self.by = np.zeros((output_size,1))

    def sigmoid(self,x):

        return 1/(1+np.exp(-x))



    def forward(self, xs):

        self.xs = xs
        self.cache = []

        self.hs = {}
        self.cs = {}
        self.ys = {}

        self.hs[-1] = np.zeros((self.hidden_size,1))

        self.cs[-1] = np.zeros((self.hidden_size,1))

        for t in range(len(xs)):

            x = xs[t]

            h_prev = self.hs[t-1]
            c_prev = self.cs[t-1]

            concat = np.vstack([h_prev,x])

            f = self.sigmoid(self.Wf @ concat + self.bf)

            i = self.sigmoid(self.Wi @ concat + self.bi)

            g = np.tanh(self.Wg @ concat + self.bg)

            o = self.sigmoid(self.Wo @ concat + self.bo)

            c = (f*c_prev +i*g)

            h = (o*np.tanh(c))

            y = (self.Why @ h +self.by)

            self.hs[t] = h
            self.cs[t] = c
            self.ys[t] = y

            self.cache.append((concat,f,i,g,o,c_prev,c))

        return self.ys



    def loss(self, targets):

        loss = 0

        for t in range(len(targets)):

            diff = (self.ys[t] -targets[t])
            loss += 0.5 * np.sum(diff**2)

        return loss



    def backward(self, targets):

        self.dWf = np.zeros_like(self.Wf)
        self.dWi = np.zeros_like(self.Wi)
        self.dWg = np.zeros_like(self.Wg)
        self.dWo = np.zeros_like(self.Wo)

        self.dbf = np.zeros_like(self.bf)
        self.dbi = np.zeros_like(self.bi)
        self.dbg = np.zeros_like(self.bg)
        self.dbo = np.zeros_like(self.bo)
        self.dWhy = np.zeros_like(self.Why)
        self.dby = np.zeros_like(self.by)
        dh_next = np.zeros((self.hidden_size,1))
        dc_next = np.zeros((self.hidden_size,1))

        T = len(targets)

        for t in reversed(range(T)):
            (concat,f,i,g,o,c_prev,c) = self.cache[t]
            h = self.hs[t]
            dy = (self.ys[t] -targets[t])
            self.dWhy += (dy @ h.T)
            self.dby += dy
            dh = (self.Why.T @ dy)
            dh += dh_next
            do = dh * np.tanh(c)
            dc = (dh *o *(1 - np.tanh(c)**2))
            dc += dc_next

            # -----------------
            # c=f*cprev+i*g
            # -----------------

            df = dc * c_prev
            di = dc * g
            dg = dc * i
            dc_next = dc * f

            # -----------------
            # gate derivatives
            # -----------------

            daf = (df *f *(1-f))
            dai = (di *i *(1-i))
            dao = (do *o *(1-o))
            dag = (dg *(1-g*g))

            # -----------------
            # parameter grads
            # -----------------

            self.dWf += (daf @ concat.T)
            self.dWi += (dai @ concat.T)
            self.dWo += (dao @ concat.T)
            self.dWg += (dag @ concat.T)

            self.dbf += daf
            self.dbi += dai
            self.dbo += dao
            self.dbg += dag

   
            dconcat = (self.Wf.T @ daf +self.Wi.T @ dai +self.Wo.T @ dao +self.Wg.T @ dag)
            dh_next = dconcat[:self.hidden_size]



    def step(self, lr):

        self.Wf -= lr*self.dWf
        self.Wi -= lr*self.dWi
        self.Wg -= lr*self.dWg
        self.Wo -= lr*self.dWo

        self.bf -= lr*self.dbf
        self.bi -= lr*self.dbi
        self.bg -= lr*self.dbg
        self.bo -= lr*self.dbo

        self.Why -= lr*self.dWhy
        self.by  -= lr*self.dby

In [142]:
dataset = []

for value in [1,2,3,4,5,6,7,8,9]:

    xs = [
        np.array([[float(value)]]),
        np.array([[0.0]]),
        np.array([[0.0]]),
        np.array([[0.0]])
    ]

    targets = [
        np.array([[0.0]]),
        np.array([[0.0]]),
        np.array([[0.0]]),
        np.array([[float(value)]])
    ]

    dataset.append((xs, targets))

In [143]:
model = LSTM(
    input_size=1,
    hidden_size=8,
    output_size=1
)

for epoch in range(500):

    total_loss = 0

    for xs, targets in dataset:

        model.forward(xs)

        loss = model.loss(targets)

        model.backward(targets)

        model.step(0.01)

        total_loss += loss

    if epoch % 100 == 0:

        print(
            epoch,
            total_loss
        )

0 134.76303959403415
100 6.481987219930205
200 6.006540814113431
300 1.9296073591394411
400 0.762881671250156


# Pytorch Implementation

In [144]:
class LSTM_Torch(nn.Module):

    def __init__(self,input_size,hidden_size,output_size):

        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        concat_size = input_size + hidden_size
        # Forget gate
        self.Wf = nn.Linear(concat_size,hidden_size)
        # Input gate
        self.Wi = nn.Linear(concat_size,hidden_size)
        # Candidate gate
        self.Wg = nn.Linear(concat_size,hidden_size)
        # Output gate
        self.Wo = nn.Linear(concat_size,hidden_size)
        # Output layer
        self.fc = nn.Linear(hidden_size,output_size)

    def forward(self, xs):

        seq_len = xs.size(0)
        batch_size = xs.size(1)

        h = torch.zeros(batch_size,self.hidden_size,device=xs.device)
        c = torch.zeros(batch_size,self.hidden_size,device=xs.device)

        outputs = []

        for t in range(seq_len):

            x = xs[t]

            concat = torch.cat([h, x],dim=1)
            f = torch.sigmoid(self.Wf(concat))
            i = torch.sigmoid(self.Wi(concat))
            g = torch.tanh(self.Wg(concat))
            o = torch.sigmoid(self.Wo(concat))
            c = f * c + i * g

            h = o * torch.tanh(c)

            y = self.fc(h)

            outputs.append(y)

        outputs = torch.stack(outputs)

        return outputs

In [145]:
model = LSTM_Torch(input_size=1,hidden_size=32,output_size=1)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)

In [146]:
def make_sample(v):

    xs = torch.tensor(
        [
            [[float(v)]],
            [[0.0]],
            [[0.0]],
            [[0.0]]
        ]
    )

    ys = torch.tensor(
        [
            [[0.0]],
            [[0.0]],
            [[0.0]],
            [[float(v)]]
        ]
    )

    return xs, ys


dataset = [
    make_sample(i)
    for i in range(1,50)
]

In [147]:
for epoch in range(1000):
    total_loss = 0

    for xs, targets in dataset:

        preds = model(xs)
        loss = criterion(preds,targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 100 == 0:
        print(epoch,total_loss)

0 9681.005203694105
100 27.114275930449367
200 3.1896011964417994
300 1.796972032461781
400 3.1748426988488063
500 6.092101227259263
600 6.018670354736969
700 3.683785682078451
800 4.109068372519687
900 1.0547917641233653


In [151]:
xs, _ = make_sample(49)

with torch.no_grad():

    preds = model(xs)

print(preds.squeeze())

tensor([-0.0705,  0.0745,  0.0646, 48.9909])
